# AMP-ML — quickstart

Antimicrobial peptide (AMP) binary classification on **4,921** sequence-derived features, selected down to **300**, evaluated on a held-out test set of **n=384** that is never used for model selection.

This notebook runs in **under a minute with no training and no data download** — it loads the shipped models and the held-out test split, then reproduces the headline results table and the negative-sample sensitivity study.

> **What is being claimed.** The pipeline is reproducible and independently auditable (a five-layer audit recomputes 30 numbers). The benchmark accuracy is **0.8646**. We do *not* claim state-of-the-art. See the [Limitations](#) section of the README for what these numbers do and do not mean — in particular, roughly 2.8 accuracy points come from the negative class being easier than the positive class.

## 1. Setup

In [ ]:
# If you are running this outside the repository, uncomment and point at your clone:
# %cd /content/ampml

import sys, subprocess

# Install the runtime dependencies (fast: the wheels are small).
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
               "scikit-learn", "xgboost", "shap", "pandas", "numpy", "matplotlib"],
               check=False)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
print("numpy", np.__version__)
print("pandas", pd.__version__)

## 2. The headline results — independent test set (n=384)

These numbers come straight from `results/model_comparison_data.csv`, which is generated by the pipeline. We read the artifact rather than recomputing, so what you see is exactly what the repository ships.

In [ ]:
comparison = pd.read_csv("results/model_comparison_data.csv")

display(comparison.style.format({
    "Accuracy": "{:.4f}", "Precision": "{:.4f}",
    "Recall": "{:.4f}", "F1-Score": "{:.4f}", "ROC-AUC": "{:.4f}",
}).hide(axis="index"))

best = comparison.loc[comparison["Accuracy"].idxmax()]
print(f"Recommended model: {best['Model']}  |  accuracy {best['Accuracy']:.4f}  |  ROC-AUC {best['ROC-AUC']:.4f}")

## 3. Miss rate — the metric that matters for screening

For a screening tool, the share of **true actives that get missed** matters more than overall accuracy. It is computed as `FN / (FN + TP)` — the denominator is the number of real positives, not the total sample count.

In [ ]:
cm = pd.read_csv("results/confusion_matrix_data.csv")
cm["miss_rate"] = cm["FN"] / (cm["FN"] + cm["TP"])

display(cm[["Model", "TN", "FP", "FN", "TP", "miss_rate"]].style.format({
    "miss_rate": "{:.2%}"
}).hide(axis="index"))

## 4. ROC curves

In [ ]:
roc = pd.read_csv("results/roc_curve_data.csv")
print(roc.columns.tolist())
print(roc.head())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for model, group in roc.groupby("Model"):
    ax.plot(group["FPR"], group["TPR"], label=model, linewidth=1.8)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4, linewidth=1)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curves — independent test set (n=384)")
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.grid(alpha=0.25, linewidth=0.5)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
plt.tight_layout()

## 5. How much of the accuracy is real?

This is the part we care most about. The same six models were trained against three different constructions of the negative set and evaluated on an **identical** test split, so the only variable is how the training negatives were built.

- **`baseline`** — the shipped UniProt fragments
- **`length_matched`** — negatives resampled to the positives' length profile
- **`shuffled_positives`** — relabelled copies of the *genuine* actives (an upper-bound probe: a model learning real antimicrobial activity must collapse here)

In [ ]:
sens = pd.read_csv("results/negative_sampling_sensitivity_summary.csv")
display(sens.style.format({
    "accuracy_mean": "{:.4f}", "accuracy_std": "{:.4f}",
    "f1_mean": "{:.4f}", "roc_auc_mean": "{:.4f}", "roc_auc_std": "{:.4f}",
    "miss_rate_mean": "{:.4f}",
}).hide(axis="index"))

In [ ]:
detail = pd.read_csv("results/negative_sampling_sensitivity.csv")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
colors = {"baseline": "#1D9E75", "length_matched": "#378ADD", "shuffled_positives": "#D85A30"}
models = list(detail["model"].unique())
width = 0.26
positions = np.arange(len(models))

for ax, metric, title in zip(axes, ["accuracy", "f1_score", "roc_auc"], ["Accuracy", "F1-score", "ROC-AUC"]):
    for i, strategy in enumerate(["baseline", "length_matched", "shuffled_positives"]):
        subset = detail[detail["strategy"] == strategy].set_index("model").reindex(models)
        ax.bar(positions + (i - 1) * width, subset[metric].to_numpy(), width,
               label=strategy, color=colors[strategy], edgecolor="white", linewidth=0.5)
    ax.set_xticks(positions)
    ax.set_xticklabels([m.replace(" ", "\n") for m in models], fontsize=8)
    ax.set_title(title)
    ax.set_ylim(0, 1.0)
    ax.grid(axis="y", alpha=0.25, linewidth=0.5)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

axes[0].legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.28))
fig.suptitle("Sensitivity of test-set performance to negative-sample construction", y=1.02)
plt.tight_layout()

**Reading the result:**

1. `shuffled_positives` collapses to chance (ROC-AUC ≈ 0.48). This is the good news: the models are learning genuine sequence content, not a label artifact — the collection artifact was still present in the relabelled data, and the models did not exploit it.
2. `length_matched` costs about **2.8 accuracy points** versus `baseline`. The L0 class (UniProt full-protein fragments) has a different length distribution from curated natural AMPs, and roughly 3 points of the headline accuracy come from that mismatch. Any AMP benchmark that does not control for this is reporting a partly inflated figure.

We publish this because we could not find another repository that quantifies it.

## 6. Score your own sequences

The `ampml` CLI scores a feature table with the trained Gradient Boosting model. For raw peptide sequences you first need the upstream descriptor extraction (see `ampmodel/feature_extraction/`).

In [ ]:
test_features = pd.read_csv("data/test_data_selected.csv")
print(test_features.shape)
print(test_features.columns[:5].tolist(), "...", test_features.columns[-1])

## 7. Reproduce everything locally

```bash
git clone https://github.com/ssinder/ampml.git
cd ampml
python3.13 -m venv venv
./venv/bin/pip install -r requirements.txt
./venv/bin/python src/00_check_environment.py   # gate 1
bash run_all.sh                                  # gate 2: full pipeline + audit
```

The five-layer audit lives in `src/99_verify_test_metrics.py` and checks 30 numerical values across data provenance, metric recomputation, and model fingerprints.

---

**License**: MIT. Please cite the repository if you use it — a `CITATION.cff` is included.